# 🤖 Fase eliminatoria - Mundial 2026: Análisis de Octavos de Final

## 1. Inicialización del Entorno y Carga de Datos
En esta sección importamos las librerías analíticas fundamentales (`pandas`, `numpy`, `scipy.stats`) y cargamos la **Matriz de Características Consolidada** (`features_rendimiento_2026.csv`). Esta matriz contiene las métricas históricas de rendimiento y las variables de forma de cada selección clasificada.

In [6]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import random
import pickle

# 1. Cargar matriz de características con ruta dinámica
ruta_features = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'processed', 'features_rendimiento_2026.csv'))
df_features = pd.read_csv(ruta_features)

# Homologación de columnas
columnas_mapeo = {
    'Equipo': 'equipo',
    'Efectividad_Historica': 'efectividad_historica_%',
    'Puntos_Por_Partido': 'puntos_por_pj_2026',
    'Expectativa_Goles_Poisson': 'expectativa_goles_poisson'
}
df_features = df_features.rename(columns=columnas_mapeo)

# 2. Cargar los resultados de Monte Carlo desde el Notebook 2
ruta_pickle = '../data/processed/montecarlo_results.pkl'
with open(ruta_pickle, 'rb') as f:
    res_montecarlo_global = pickle.load(f)

print("✅ Entorno sincronizado: Características y resultados de Monte Carlo cargados.")

✅ Entorno sincronizado: Características y resultados de Monte Carlo cargados.


## 2. Definición del Cuadro del Torneo y Cruce de Variables
Estructuramos las 8 llaves oficiales de los Octavos de Final del Mundial. Mediante operaciones de unión (*merge*), cruzamos los datos de rendimiento de los equipos locales (`home_team`) y visitantes (`away_team`) para calcular los primeros deltas teóricos de rendimiento.

In [7]:
# 1. Definición del cuadro de Octavos de Final (Actualizado con ganadores reales)
# Cuadro oficial tras los partidos del 3 de julio
octavos_proyectados = [
    {"home_team": "Paraguay", "away_team": "France"},
    {"home_team": "Canada", "away_team": "Morocco"},
    {"home_team": "Brazil", "away_team": "Norway"},
    {"home_team": "Mexico", "away_team": "England"},
    {"home_team": "Portugal", "away_team": "Spain"}, 
    {"home_team": "United States", "away_team": "Belgium"},
    {"home_team": "Argentina", "away_team": "Egypt"},
    {"home_team": "Switzerland", "away_team": "Colombia"}
]

df_r16 = pd.DataFrame(octavos_proyectados)
print(f"✅ Cuadro de Octavos cargado con {len(df_r16)} partidos oficiales.")

✅ Cuadro de Octavos cargado con 8 partidos oficiales.


## 3. Análisis Exploratorio: Gradiente de Forma Reciente
Antes de correr las simulaciones matemáticas, desplegamos un gráfico divergente utilizando `Seaborn`. Este mapa visual nos muestra qué equipos llegan con mayor inercia ganadora en el torneo actual evaluando el diferencial de puntos obtenidos por partido.

In [8]:
# 1. Preparación estética y ordenamiento predictivo de los datos
plt.figure(figsize=(13, 7))
sns.set_theme(style="whitegrid")

df_octavos_crossed['llave'] = df_octavos_crossed['home_team'] + " vs " + df_octavos_crossed['away_team']
df_grafico = df_octavos_crossed.sort_values(by='diff_forma_2026', ascending=False)

# 2. Despliegue del gráfico divergente con Seaborn utilizando el valor numérico para mapear colores
sns.barplot(
    data=df_grafico, 
    x='diff_forma_2026', 
    y='llave', 
    palette='coolwarm',
    hue='diff_forma_2026',
    legend=False
)

plt.axvline(x=0, color='black', linestyle='--', linewidth=1.2)
plt.title('Divergencia Teórica de Forma Reciente (Puntos por Partido en 2026)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Delta de Forma (Local - Visitante)', fontsize=11)
plt.ylabel('Cruce de Octavos', fontsize=11)
plt.tight_layout()
plt.show()

NameError: name 'df_octavos_crossed' is not defined

<Figure size 1300x700 with 0 Axes>

## 4. Modelado Predictivo: Distribución de Poisson y Simulaciones de Monte Carlo
Esta sección aloja el núcleo del algoritmo predictivo:
1. **Fuerzas Combinadas:** Balanceamos un 60% el momento actual en el mundial y un 40% el peso de la historia para ajustar las expectativas de gol ($\lambda$).
2. **Distribución de Poisson:** Evaluamos probabilísticamente los escenarios de prórroga y penaltis.
3. **Generador Estocástico Unitario:** Proyecta un marcador exacto con minutos de goles ponderados para el segundo tiempo.
4. **Simulación de Monte Carlo:** Ejecuta **1,000 torneos alternativos en paralelo** utilizando vectores de NumPy para obtener la probabilidad real de supervivencia/clasificación de cada país.

In [ ]:
import scipy.stats as stats
import random
import numpy as np

# ==============================================================================
# 5. MODELADO PREDICTIVO: PROBABILIDADES DE ALARGUE, PENALTIS Y MARCADOR EXACTO
# ==============================================================================

GOLES_PROMEDIO_TORNEO = 2.6

def calcular_y_simular_encuentro(equipo_home, equipo_away):
    dict_stats = df_features.set_index('equipo').to_dict('index')
    
    if equipo_home not in dict_stats or equipo_away not in dict_stats:
        return None
        
    feat_h, feat_a = dict_stats[equipo_home], dict_stats[equipo_away]
    
    # 🔄 AJUSTE DE PESOS: Consistente con el Notebook 2 (70% forma 2026 / 30% Histórica)
    fuerza_h = (feat_h['puntos_por_pj_2026'] * 0.7) + ((feat_h['efectividad_historica_%'] / 100) * 0.3)
    fuerza_a = (feat_a['puntos_por_pj_2026'] * 0.7) + ((feat_a['efectividad_historica_%'] / 100) * 0.3)
    
    # Normalización de fuerzas para Poisson
    sum_fuerzas = fuerza_h + fuerza_a
    lambda_h = max(0.4, min((fuerza_h / sum_fuerzas) * GOLES_PROMEDIO_TORNEO * 1.5, 2.5))
    lambda_a = max(0.4, min((fuerza_a / sum_fuerzas) * GOLES_PROMEDIO_TORNEO * 1.5, 2.5))
    
    max_goles = 9
    prob_goles_h = [stats.poisson.pmf(g, lambda_h) for g in range(max_goles)]
    prob_goles_a = [stats.poisson.pmf(g, lambda_a) for g in range(max_goles)]
    prob_tiempo_extra = sum(prob_goles_h[g] * prob_goles_a[g] for g in range(max_goles))
    
    goles_h, goles_a = stats.poisson.rvs(lambda_h), stats.poisson.rvs(lambda_a)
    
    return {
        "goles_h": goles_h, "goles_a": goles_a,
        "prob_te": prob_tiempo_extra * 100,
        "lambda_h": lambda_h, "lambda_a": lambda_a,
        "fuerza_h": fuerza_h, "fuerza_a": fuerza_a
    }

# ==============================================================================
# MOTOR DE SIMULACIÓN DE MONTE CARLO (1,000 ITERACIONES VECTORIZADAS)
# ==============================================================================
def simular_monte_carlo(equipo_home, equipo_away, n_simulaciones=1000):
    base_stats = calcular_y_simular_encuentro(equipo_home, equipo_away)
    if not base_stats: return None
    
    # Generación masiva vectorizada en memoria
    goles_sim_h = stats.poisson.rvs(base_stats["lambda_h"], size=n_simulaciones)
    goles_sim_a = stats.poisson.rvs(base_stats["lambda_a"], size=n_simulaciones)
    
    # Conteo de clasificaciones directas
    clasifica_home = (goles_sim_h > goles_sim_a).sum()
    clasifica_away = (goles_sim_a > goles_sim_h).sum()
    empates = (goles_sim_h == goles_sim_a).sum()
    
    # Desempate de penaltis mediante distribución binomial masiva
    prob_desempate_home = base_stats["fuerza_h"] / (base_stats["fuerza_h"] + base_stats["fuerza_a"])
    desempates_home = np.random.binomial(empates, prob_desempate_home)
    
    total_home_win = clasifica_home + desempates_home
    total_away_win = clasifica_away + (empates - desempates_home)
    
    return {
        "pct_home": (total_home_win / n_simulaciones) * 100,
        "pct_away": (total_away_win / n_simulaciones) * 100
    }

# --- RUNTIME GLOBAL DE ANÁLISIS ---
print("====== 🏆 PROYECCIÓN ESTADÍSTICA AVANZADA (OCTAVOS) ======\n")
res_montecarlo_global = {}

for _, partido in df_r16.iterrows():
    home, away = partido['home_team'], partido['away_team']
    res = calcular_y_simular_encuentro(home, away)
    mc = simular_monte_carlo(home, away, n_simulaciones=1000)
    
    if res and mc:
        res_montecarlo_global[f"{home} vs {away}"] = mc
        print(f"🏟️  {home} vs {away}")
        print(f"    🎲 Monte Carlo: Clasificación {home}: {mc['pct_home']:.1f}% | {away}: {mc['pct_away']:.1f}%")
        print(f"    🎴 Marcador Único (90'): {home} {res['goles_h']} - {res['goles_a']} {away}")
        print("-" * 70)

====== 🏆 PROYECCIÓN ESTADÍSTICA AVANZADA Y MONTE CARLO (OCTAVOS) ======

🏟️  Paraguay vs France
    📊 Probabilidad de Tiempo Extra: 21.54%
    🎯 Probabilidad de llegar a Penaltis: 11.16%
    🎲 Simulación Monte Carlo (1,000 ejecuciones):
        ➡️ Probabilidad Clasificación Paraguay: 26.6%
        ➡️ Probabilidad Clasificación France: 73.4%
    🎴 Marcador Único (90'): Paraguay 1 - 2 France
----------------------------------------------------------------------
🏟️  Canada vs Morocco
    📊 Probabilidad de Tiempo Extra: 25.32%
    🎯 Probabilidad de llegar a Penaltis: 13.73%
    🎲 Simulación Monte Carlo (1,000 ejecuciones):
        ➡️ Probabilidad Clasificación Canada: 39.9%
        ➡️ Probabilidad Clasificación Morocco: 60.1%
    🎴 Marcador Único (90'): Canada 2 - 1 Morocco
----------------------------------------------------------------------
🏟️  Brazil vs Norway
    📊 Probabilidad de Tiempo Extra: 21.73%
    🎯 Probabilidad de llegar a Penaltis: 11.28%
    🎲 Simulación Monte Carlo (1,000 

## 3. Análisis Exploratorio: Gradiente de Forma Reciente
Antes de correr las simulaciones matemáticas, desplegamos un gráfico divergente utilizando `Seaborn`. Este mapa visual nos muestra qué equipos llegan con mayor inercia ganadora en el torneo actual evaluando el diferencial de puntos obtenidos por partido, permitiendo un desglose preliminar del cruce entre **México** e **Inglaterra**.

In [ ]:
import pandas as pd
import os

# 1. Definir el encuentro
equipo_home = "Portugal"
equipo_away = "Spain"
llave = f"{equipo_home} vs {equipo_away}"

# 2. Recuperar probabilidades desde el diccionario global
pct_home = res_montecarlo_global[llave]["pct_home"]
pct_away = res_montecarlo_global[llave]["pct_away"]

# 3. Extraer stats automáticamente desde df_features
def get_stats(equipo):
    row = df_features[df_features['equipo'] == equipo].iloc[0]
    return row['efectividad_historica_%'], row['puntos_por_pj_2026'], row['expectativa_goles_poisson']

eff_h, pts_h, gol_h = get_stats(equipo_home)
eff_a, pts_a, gol_a = get_stats(equipo_away)

# 4. Estructura del DataFrame
match_data = {
    'Equipo': [equipo_home, equipo_away],
    'Probabilidad_Clasificacion': [pct_home, pct_away],
    'Efectividad_Historica': [eff_h, eff_a],
    'Puntos_Por_Partido': [pts_h, pts_a],
    'Expectativa_Goles_Poisson': [gol_h, gol_a]
}

df_match = pd.DataFrame(match_data)

# 5. Formatear para Tableau (comas en decimales)
columnas_num = ['Probabilidad_Clasificacion', 'Efectividad_Historica', 'Puntos_Por_Partido', 'Expectativa_Goles_Poisson']
for col in columnas_num:
    df_match[col] = df_match[col].round(2).astype(str).str.replace('.', ',', regex=False)

# 6. Guardar en ruta dinámica
ruta_output = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'processed', f'Reporte_{equipo_home}_{equipo_away}.csv'))

os.makedirs(os.path.dirname(ruta_output), exist_ok=True)
df_match.to_csv(ruta_output, index=False, encoding='utf-8-sig', sep=';')

print(f"✅ ¡Archivo exportado para {llave}!")
print(f"📍 Destino: {ruta_output}")

KeyError: 'Portugal vs Spain'